In [1]:
#Script 5/5 — dim_sales_reps
#Generates 75 rows.
#Injects: nulls, mixed boolean for is_active, mixed date formats,
#         near-duplicate names, inconsistent rep_id prefixes, outlier hire dates.

In [2]:
import os, random
import pandas as pd
import numpy as np
from faker import Faker

In [3]:
fake = Faker()
random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [4]:
os.makedirs("data/raw_1", exist_ok=True)

In [5]:
N = 75

TEAMS    = ["Enterprise", "SMB", "Online", "Retail", "Partner"]
MANAGERS = [fake.name() for _ in range(8)]   # pool of 8 managers

In [6]:
def mixed_date(dt, style):
    ts = pd.Timestamp(dt)
    if style == 0:
        return ts.strftime("%m/%d/%Y")
    elif style == 1:
        return ts.strftime("%Y-%m-%d")
    else:
        return ts.strftime("%d-%b-%Y")

In [7]:
def mixed_bool(val):
    style = random.randint(0, 2)
    if style == 0:
        return "True" if val else "False"
    elif style == 1:
        return "YES" if val else "NO"
    else:
        return 1 if val else 0

records = []
for i in range(1, N + 1):
    hire_date = fake.date_between(start_date="-12y", end_date="today")
    is_act    = random.choices([True, False], [0.85, 0.15])[0]
    d_style   = random.randint(0, 2)

    records.append({
        "rep_id":    f"REP-{str(i).zfill(3)}",
        "rep_name":  fake.name(),
        "team":      random.choice(TEAMS),
        "manager":   random.choice(MANAGERS),
        "hire_date": mixed_date(hire_date, d_style),
        "is_active": mixed_bool(is_act),
    })

In [8]:
df = pd.DataFrame(records)

In [9]:
# PART 2 - Data quality injections

In [10]:
# a) Null

In [11]:
for col, rate in [("manager", 0.10), ("team", 0.12),
                  ("hire_date", 0.11)]:
    null_idx = df.sample(frac=rate, random_state=42).index
    df.loc[null_idx, col] = np.nan

In [12]:
# b) Near duplicates: same rep, name, spacing/casing tweaks

In [13]:
dup_idx = df.sample(frac=0.04, random_state=7).index
dups    = df.loc[dup_idx].copy()
dups["rep_name"] = dups["rep_name"].str.lower().apply(lambda x: "  " + x + "  ")
dups["rep_id"]   = dups["rep_id"].str.replace("REP-", "R-", regex=False)
df = pd.concat([df, dups], ignore_index=True)

In [14]:
# c) Inconsistent rep_id prefixes

In [15]:
prefix_idx = df.sample(frac=0.06, random_state=55).index
df.loc[prefix_idx, "rep_id"] = (
    df.loc[prefix_idx, "rep_id"]
      .str.replace("REP-", "", regex=False)
      .str.lstrip("0")
      .apply(lambda x: f"SR{x}")
)

In [16]:
# d) Outlier: future hire dates

In [17]:
future_idx = df.sample(3, random_state=33).index
df.loc[future_idx, "hire_date"] = "2028-01-01"

In [18]:
# PART 3

In [19]:
# 3-5 rows where end_date < start_date modelled as
#     hire_date > today (future dates above cover this pattern)
#     Additionally inject an "end_date" column with 4 rows violated

In [20]:
df["end_date"] = pd.NaT
active_idx = df[df["is_active"].isin(["False", "NO", 0])].sample(
    min(6, df[df["is_active"].isin(["False", "NO", 0])].shape[0]),
    random_state=42
).index
for idx in active_idx:
    h = df.loc[idx, "hire_date"]
    try:
        ts = pd.Timestamp(h)
        df.loc[idx, "end_date"] = ts + pd.Timedelta(days=random.randint(180, 1800))
    except Exception:
        pass

# Inject 4 violations: end_date < hire_date
violation_idx = df.dropna(subset=["end_date"]).sample(
    min(4, df.dropna(subset=["end_date"]).shape[0]), random_state=77
).index
for idx in violation_idx:
    h = df.loc[idx, "hire_date"]
    try:
        ts = pd.Timestamp(h)
        df.loc[idx, "end_date"] = ts - pd.Timedelta(days=random.randint(30, 365))
    except Exception:
        pass

In [21]:
# Shuffle & save

In [22]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
out_path = "data/raw_1/dim_sales_reps.csv"
df.to_csv(out_path, index=False)
print(f"✅ dim_sales_reps saved → {out_path}  |  shape: {df.shape}")
print(df.dtypes)
print(df.head(3))
print("\nend_date < hire_date violations:")
temp = df.dropna(subset=["end_date", "hire_date"]).copy()
temp["hire_dt"] = pd.to_datetime(temp["hire_date"], errors="coerce")
temp["end_dt"]  = pd.to_datetime(temp["end_date"],  errors="coerce")
print(temp[temp["end_dt"] < temp["hire_dt"]][["rep_id","hire_date","end_date"]])

✅ dim_sales_reps saved → data/raw_1/dim_sales_reps.csv  |  shape: (78, 7)
rep_id               object
rep_name             object
team                 object
manager              object
hire_date            object
is_active            object
end_date     datetime64[ns]
dtype: object
    rep_id        rep_name team          manager    hire_date is_active  \
0  REP-034  Michael Brooks  SMB  Cristian Santos  16-Sep-2019         1   
1  REP-001    James Howard  NaN              NaN          NaN      True   
2  REP-035     David Brown  SMB    Daniel Wagner   2023-03-19         0   

    end_date  
0        NaT  
1        NaT  
2 2022-08-18  

end_date < hire_date violations:
     rep_id    hire_date   end_date
2   REP-035   2023-03-19 2022-08-18
51     SR16  16-Dec-2023 2023-10-27
56  REP-025  20-Mar-2016 2015-09-15
60  REP-012   05/11/2015 2014-10-07
